# Process Forcings (`case.process_forcings`)

The final step: generate the forcing files your `configure_forcings` call (previous
notebook) declared. This notebook covers the normal one-shot path, and the
offline/iterative path for long or large runs where generating everything at once is
too slow or resource-intensive.

This notebook covers:
- [Section 1](#section-1-normal-one-shot-processing) — the normal, one-shot path
- [Section 2](#section-2-offline-or-iterative-processing-for-long-runs) — offline/iterative processing (`config.json`, `driver.py`) for long runs

📖 [CrocoDash process_forcings docs](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/3b_process_forcings.html) · [CLI reference](advanced/cli_workflow.md) · [regional-mom6 docs](https://regional-mom6.readthedocs.io/en/latest/) (OBC regridding internals)

## Section 1: Normal, One-Shot Processing

In this step, we call the `process_forcings` method of CrocoDash to cut out and
interpolate the initial condition as well as all boundaries. CrocoDash also updates
MOM6 runtime parameters and CESM xml variables accordingly.

In [ ]:
case.process_forcings()

In [ ]:
print("You can now build and run your case at", caseroot)

## Section 2: Offline or Iterative Processing (for Long Runs)

Often, generating OBC datasets can be done all in one shot, but in longer and larger
cases (like running the Northwest Atlantic for a year) we need to start iterating
through the generation. Generating open boundary condition (OBC) data is essential for
the entire model runtime but can be time-consuming and resource-intensive.

The Extract Forcings Workflow in CrocoDash helps manage this by breaking data access
into smaller, more manageable components.

### Extract Forcings Workflow Overview

The workflow is enabled in all cases. When `configure_forcings` is called, it triggers
copying of a config folder into the case input directory's forcing folder, and the
generation of a configuration file (`config.json`) to download the required boundary
condition and other forcing files. You can trigger the workflow from the shell with
`crocodash process --all` (recommended — see the [CLI reference](advanced/cli_workflow.md)),
or by running `driver.py` directly inside the `extract_forcings` folder and adjusting
config options from the command line or in Python.

#### Folder Structure

- **config.json** – Defines the region-specific requirements and run parameters.
- **README** – Explains the workflow.
- **driver.py** – Executes all scripts needed to obtain OBC data; this is what
  `crocodash process` calls under the hood.
- **raw_data/**, **regridded_data/** – Intermediate storage for workflow steps,
  preventing the need to rerun all scripts at once.

#### Scripts — not in the config folder, but in CrocoDash under `extract_forcings`

1. **get_data_piecewise** – Retrieves raw, unprocessed data in chunks (size defined by
   `config["conditions"]["outputs"]["step"]`) and saves it to `raw_data/`.
2. **regrid_data_piecewise** – Processes raw data and stores it in `regridded_data/`.
3. **merge_piecewise_dataset** – Combines regridded data into the final dataset for
   model input.
4. **Specific Forcing Functions** – chlorophyll, tides, runoff, bgc, ...

#### How to Use

1. Identify and allocate available computing resources.
2. Adjust the `step` parameter (inside `config.json`'s `conditions.outputs` block) to
   match resource constraints — default is 5 days. Smaller steps mean more, cheaper
   chunks; this is the main knob for fitting a long run into a walltime or memory limit.
3. Run each step manually, or use `driver.py`/`crocodash process` as a guide — you can
   rerun with a different `--skip` list if only some forcings changed, without redoing
   everything.

See the CrocoDash docs' [Process Forcings](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/3b_process_forcings.html)
page for the full pipeline diagram and CLI flag reference.

### Running the Iterative OBC Processor

The `extract_forcings` folder is a **self-contained copy** placed under your input
directory's forcing folder (default: `glorys/extract_forcings`) — you can run it
standalone from the shell without importing CrocoDash again:

```bash
cd <inputdir>/glorys/extract_forcings
python driver.py --all
```

Or, with CrocoDash installed, from anywhere:

```bash
crocodash process --caseroot <caseroot> --all
```

Both read `config.json` and generate the OBCs piecewise before merging — modify either
the config or the code as you see fit.

Especially consider adjusting the data-download function in `config.json`: on Derecho,
use the RDA reader (`get_glorys_data_from_rda`); on a local machine, use a GLORYS
API/CLI function instead (see [Configure Forcings, Section 2](configure_forcings.ipynb#section-2-switching-data-products)
for the full list of access functions). Change it by editing the `function_name` field
under `config.json`'s `conditions.inputs` block.

Use `--help` (or `crocodash process --help`) to see all available flags, including
running only specific forcings (`--tides --runoff --bgc`) or skipping ones you don't
need (`--all --skip bgcic`).

## Next steps

- [Interior OBC Segments](advanced/interior_obc_segments.ipynb) and
  [Nesting](advanced/nesting_demo.ipynb) — advanced domain/boundary topics
- [Use Cases](use_cases/three_boundary.ipynb) — real-world, end-to-end configurations